In [ ]:

from pathlib import Path
import pandas as pd

PROJECT_ROOT = next(
    (candidate for candidate in [Path.cwd(), *Path.cwd().parents]
     if (candidate / "data/Labor_rate_data.csv").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Cannot locate data/Labor_rate_data.csv from this working directory.")
RAW_DATA_PATH = PROJECT_ROOT / "data/Labor_rate_data.csv"
CLEAN_TRACKER_PATH = PROJECT_ROOT / "data/cleaned_raw_labor_rates_tracker.csv"
VALIDATION_RATES_PATH = PROJECT_ROOT / "data/labor_rates_fa.csv"

if not RAW_DATA_PATH.exists():
    raise FileNotFoundError(f"Source dataset not found: {RAW_DATA_PATH.resolve()}")

print(f"Source dataset: {RAW_DATA_PATH.resolve()}")
print(f"Cleaned tracker will be saved to: {CLEAN_TRACKER_PATH}")
print(f"Validation table will be saved to: {VALIDATION_RATES_PATH}")

In [ ]:
# load in dataset. Contains headers (which we will keep for now) and make sure all types are strings incase of errors
raw_df = pd.read_csv(
    RAW_DATA_PATH,
    header=None,
    dtype=str,
    keep_default_na=False,
)

print(f"Raw dataset dimensions: {raw_df.shape[0]} rows x {raw_df.shape[1]} columns")
display(raw_df.head(8))

In [ ]:
# Step 3: Change column names
# Unnamed source columns are retained temporarily so they can be assessed before removal.
column_names = [
    "subcontractor",                         
    "position",                              
    "unnamed_02",                            # Changed: unnamed source column -> needs more investigating if !=NAN.
    "effective_time_period_start",          
    "effective_time_period_finish",          
    "current_expired",                       
    "days_left",                             
    "unnamed_07",                            # Changed: unnamed source column -> needs more investigating if !=NAN.
    "per_sub_reg_rate",                   
    "per_sub_ot_rate",                       
    "per_sub_dt_rate",                       
    "unnamed_11",                            # Changed: unnamed source column -> needs more investigating if !=NAN
    "reg_rate",                     # Changed: unlabeled Reg group -> reg_rate
    "ot_rate",                      # Changed: unlabeled OT group -> ot_rate
    "dt_rate",                      # Changed: unlabeled DT group -> dt_rate
    "unnamed_15",                            # Changed: unnamed source column -> needs more investigating if !=NAN
    "variance_reg",                          
    "variance_ot",                           
    "variance_dt",                           
    "unnamed_19",                            # Changed: unnamed source column -> needs more investigating if !=NAN
    "rate_agreed_to_by_sub",                
    "ot_premium_only",                      
    "submitted_to_cbiz_for_approval",        
    "status",                               
    "submitted_to_leftfield_for_approvals", 
    "bu_approval_received",                 
    "status_notes",                
    "unnamed_27",                            # Changed: unnamed source column -> needs more investigating if !=NAN
    "unnamed_28",                            # Changed: unnamed source column -> needs more investigating if !=NAN
]

# The first three rows contain grouped headings, field headings, and a blank separator.
# Remove those non-record rows and apply the documented schema to the working copy.
if raw_df.shape[1] != len(column_names):
    raise ValueError(
        f"Expected {len(column_names)} source columns, found {raw_df.shape[1]}."
    )

labor_rates_df = raw_df.iloc[3:].copy()
labor_rates_df.columns = column_names
labor_rates_df.reset_index(drop=True, inplace=True)

print(f"Working dataset dimensions: {labor_rates_df.shape[0]} rows x {labor_rates_df.shape[1]} columns")
display(labor_rates_df.head())

In [ ]:
# Step 4: Profile data quality before removing or modifying any records or columns.
# Treat whitespace-only strings as blank so the measurements reflect meaningful content.
blank_mask = labor_rates_df.apply(lambda column: column.str.strip().eq(""))

# Summarize completeness and distinct nonblank values for every client-aligned field.
column_profile = pd.DataFrame({
    "nonblank_count": (~blank_mask).sum(),
    "blank_count": blank_mask.sum(),
    "blank_percent": (blank_mask.mean() * 100).round(2),
    "unique_nonblank_count": [
        labor_rates_df.loc[~blank_mask[column], column].nunique()
        for column in labor_rates_df.columns
    ],
})

completely_blank_rows = blank_mask.all(axis=1)
completely_empty_columns = column_profile.index[
    column_profile["nonblank_count"].eq(0)
].tolist()
duplicate_row_count = int(labor_rates_df.duplicated().sum())

print(f"Completely blank rows: {int(completely_blank_rows.sum())}")
print(f"Duplicate rows: {duplicate_row_count}")
print(f"Completely empty columns: {completely_empty_columns}")
display(column_profile)

# Keep the client's original status wording visible while checking category consistency.
for status_column in ["current_expired", "status"]:
    print(f"\nValue counts for {status_column}:")
    display(
        labor_rates_df[status_column]
        .str.strip()
        .replace("", "<blank>")
        .value_counts(dropna=False)
    )

In [ ]:
# Step 5: check empty columns before removing them
populated_unnamed_columns = ["unnamed_15", "unnamed_28"]
populated_unnamed_mask = labor_rates_df[populated_unnamed_columns].apply(
    lambda column: column.str.strip().ne("")
).any(axis=1)

unnamed_value_context_columns = [
    "subcontractor",
    "position",
    "effective_time_period_start",
    "effective_time_period_finish",
    "reg_rate",
    "ot_rate",
    "dt_rate",
    "unnamed_15",
    "status",
    "status_notes",
    "unnamed_28",
]

unnamed_value_review = labor_rates_df.loc[
    populated_unnamed_mask, unnamed_value_context_columns
].copy()

print(f"Rows requiring unnamed-column review: {len(unnamed_value_review)}")
for column in populated_unnamed_columns:
    values = labor_rates_df.loc[
        labor_rates_df[column].str.strip().ne(""), column
    ].tolist()
    print(f"{column}: {values}")

display(unnamed_value_review)

In [ ]:
# Step 6: Resolve the two populated unnamed source columns using the completed review.
# unnamed_28 is kept and renamed because its values equal approved DT minus approved Reg.
# unnamed_15 is dropped because its lone value has no verifiable business meaning.
# Source-specific review details are kept in the ignored data/data_quality_notes.md.
labor_rates_df.rename(
    columns={"unnamed_28": "dt_premium_only"},
    inplace=True,
)
labor_rates_df.drop(columns=["unnamed_15"], inplace=True)

print("Renamed unnamed_28 to dt_premium_only.")
print("Dropped unnamed_15 after documenting its lone ambiguous value.")
print(f"Working dataset dimensions: {labor_rates_df.shape[0]} rows x {labor_rates_df.shape[1]} columns")

In [ ]:
# Step 7: Remove empty rows and columns.
# Preserve the original CSV row number so each cleaned record remains traceable.
business_columns = labor_rates_df.columns.tolist()
business_blank_mask = labor_rates_df[business_columns].apply(
    lambda column: column.str.strip().eq("")
)
blank_rows_to_remove = business_blank_mask.all(axis=1)
empty_columns_to_remove = [
    column for column in business_columns if business_blank_mask[column].all()
]

# Data begins on CSV row 4 because the first three rows are source headings/separators.
labor_rates_df.insert(
    0,
    "source_csv_row_number",
    labor_rates_df.index + 4,
)
labor_rates_df = (
    labor_rates_df.loc[~blank_rows_to_remove]
    .drop(columns=empty_columns_to_remove)
    .reset_index(drop=True)
)

print(f"Removed {int(blank_rows_to_remove.sum())} completely blank rows.")
print(f"Removed {len(empty_columns_to_remove)} completely empty columns: {empty_columns_to_remove}")
print(f"Working dataset dimensions: {labor_rates_df.shape[0]} rows x {labor_rates_df.shape[1]} columns")

In [ ]:
# Step 8: Normalize whitespace and represent missing business values consistently.
# Do not fill or remove records here; missing values may indicate issues that reviewers need to see.
text_columns = [
    column for column in labor_rates_df.columns
    if column != "source_csv_row_number"
]

labor_rates_df[text_columns] = labor_rates_df[text_columns].apply(
    lambda column: (
        column.astype("string")
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
        .replace("", pd.NA)
    )
)

missing_value_summary = (
    labor_rates_df.isna()
    .sum()
    .rename("missing_count")
    .to_frame()
)
missing_value_summary["missing_percent"] = (
    missing_value_summary["missing_count"] / len(labor_rates_df) * 100
).round(2)
missing_value_summary = missing_value_summary.loc[
    missing_value_summary["missing_count"].gt(0)
].sort_values("missing_count", ascending=False)

print("Whitespace normalized and empty strings converted to pd.NA.")
print(f"Rows retained: {len(labor_rates_df)}")
display(missing_value_summary)

In [ ]:
# Step 9: Identify likely contractor-section headers before any forward fill.
# Gaps in source row numbers preserve the blank separators removed in Step 7.
source_row_gap = labor_rates_df["source_csv_row_number"].diff().fillna(2)
section_start_mask = (
    source_row_gap.gt(1)
    & labor_rates_df["subcontractor"].notna()
)

contractor_section_candidates = labor_rates_df.loc[
    section_start_mask,
    [
        "source_csv_row_number",
        "subcontractor",
        "position",
        "effective_time_period_start",
        "reg_rate",
        "status",
        "status_notes",
    ],
].copy()

# These prefixes identify obvious context labels or contacts, not contractor names.
# The labels remain in the review table and are not discarded or reclassified automatically.
context_label_pattern = (
    r"^(?:\(|OCIP\b|Equipment\b|Lab foremen\b|Carp foremen\b|"
    r"Local \d+\b|See Notes\b)"
)
contractor_section_candidates["review_hint"] = "possible_contractor"
context_candidate_mask = contractor_section_candidates["subcontractor"].str.contains(
    context_label_pattern,
    case=False,
    regex=True,
    na=False,
)
contractor_section_candidates.loc[
    context_candidate_mask, "review_hint"
] = "likely_context_or_contact"

print(f"Section-start candidates found: {len(contractor_section_candidates)}")
print("Review hints are diagnostic only; no contractor values were filled.")
display(contractor_section_candidates)

In [ ]:
# Step 10: Carry contractor names from section headers through their rate records.
# The client's Subcontractor field also contains contacts and section notes.
# Keep that original text as subcontractor_context and add a clean subcontractor field.
contractor_header_rows = contractor_section_candidates.loc[
    contractor_section_candidates["review_hint"].eq("possible_contractor"),
    "source_csv_row_number",
]

labor_rates_df.rename(
    columns={"subcontractor": "subcontractor_context"},
    inplace=True,
)
contractor_header_mask = labor_rates_df["source_csv_row_number"].isin(
    contractor_header_rows
)
labor_rates_df.insert(
    labor_rates_df.columns.get_loc("subcontractor_context"),
    "subcontractor",
    labor_rates_df["subcontractor_context"].where(contractor_header_mask).ffill(),
)

# Every retained record should now inherit the nearest preceding contractor header.
unassigned_rows = int(labor_rates_df["subcontractor"].isna().sum())
if unassigned_rows:
    raise ValueError(f"{unassigned_rows} records have no contractor section.")

print(f"Contractor section headers used: {len(contractor_header_rows)}")
print(f"Rows assigned a subcontractor: {len(labor_rates_df)}")
print(f"Distinct subcontractors: {labor_rates_df['subcontractor'].nunique()}")
display(labor_rates_df.loc[
    labor_rates_df["subcontractor"].eq("BrandSafway Services"),
    ["source_csv_row_number", "subcontractor", "subcontractor_context", "position"],
].head(8))

In [ ]:
# Step 11: Assign consistent types to fields with a single clear meaning.
# Keep the client's field names; only remove currency formatting from approved amounts.
date_columns = [
    "effective_time_period_start",
    "effective_time_period_finish",
    "rate_agreed_to_by_sub",
    "bu_approval_received",
]
numeric_rate_columns = [
    "reg_rate",
    "ot_rate",
    "dt_rate",
    "dt_premium_only",
]

# Check for invalid dates before replacing the original strings.
for column in date_columns:
    converted = pd.to_datetime(
        labor_rates_df[column], format="%m/%d/%Y", errors="coerce"
    )
    invalid = labor_rates_df[column].notna() & converted.isna()
    if invalid.any():
        rows = labor_rates_df.loc[invalid, "source_csv_row_number"].tolist()
        raise ValueError(f"Invalid {column} dates in source CSV rows: {rows}")
    labor_rates_df[column] = converted

# Remove dollar signs and thousands separators, then store rates as nullable numbers.
for column in numeric_rate_columns:
    cleaned = labor_rates_df[column].str.replace(r"[$,]", "", regex=True)
    converted = pd.to_numeric(cleaned, errors="coerce").astype("Float64")
    invalid = labor_rates_df[column].notna() & converted.isna()
    if invalid.any():
        rows = labor_rates_df.loc[invalid, "source_csv_row_number"].tolist()
        raise ValueError(f"Invalid {column} amounts in source CSV rows: {rows}")
    labor_rates_df[column] = converted

# Days Left is a whole-number snapshot field; retain it for now as a nullable integer.
converted_days = pd.to_numeric(labor_rates_df["days_left"], errors="coerce")
invalid_days = labor_rates_df["days_left"].notna() & converted_days.isna()
if invalid_days.any():
    rows = labor_rates_df.loc[invalid_days, "source_csv_row_number"].tolist()
    raise ValueError(f"Invalid days_left values in source CSV rows: {rows}")
labor_rates_df["days_left"] = converted_days.astype("Int64")
labor_rates_df["source_csv_row_number"] = labor_rates_df["source_csv_row_number"].astype("Int64")

# These client fields mix dates, labels, or multiple premiums, so preserve them as strings.
mixed_text_columns = [
    "submitted_to_cbiz_for_approval",
    "submitted_to_leftfield_for_approvals",
    "ot_premium_only",
]
labor_rates_df[mixed_text_columns] = labor_rates_df[mixed_text_columns].astype("string")

# Report impossible effective periods as source-data issues for later review.
reversed_periods = labor_rates_df.loc[
    labor_rates_df["effective_time_period_finish"].lt(
        labor_rates_df["effective_time_period_start"]
    ),
    ["source_csv_row_number", "subcontractor", "position",
     "effective_time_period_start", "effective_time_period_finish"],
]
print(f"Rows with effective finish before start: {len(reversed_periods)}")
display(reversed_periods)

print("Column types after normalization:")
print(labor_rates_df.dtypes.to_string())

In [ ]:
# Step 12: Review the client's position names and possible terminology variations.
# Keep contractor names, position values, employee qualifiers, and adjustments as supplied.
position_name_review = labor_rates_df.loc[
    labor_rates_df["position"].notna(),
    ["source_csv_row_number", "subcontractor", "position"],
].copy()

position_inventory = (
    position_name_review.groupby("position", sort=True)
    .agg(record_count=("position", "size"),
         subcontractor_count=("subcontractor", "nunique"))
    .reset_index()
)

# Build a temporary comparison key to reveal case and common title variations.
# Foreman -> foreperson and Journeyman/Journeyperson -> journey person apply ONLY to this key.
# Parentheses, employee names, Blended, percentages, and other qualifiers remain in the key.
position_name_review["comparison_key"] = (
    position_name_review["position"].str.casefold()
    .str.replace(r"\bforeman\b", "foreperson", regex=True)
    .str.replace(r"\b(?:journeyman|journey\s*person)\b", "journey person", regex=True)
)
position_variations = (
    position_name_review.groupby("comparison_key", sort=True)
    .agg(source_names=("position", lambda values: sorted(values.unique())),
         distinct_name_count=("position", "nunique"),
         source_csv_rows=("source_csv_row_number", lambda values: values.tolist()))
    .reset_index()
)
position_variations = position_variations.loc[
    position_variations["distinct_name_count"].gt(1)
].reset_index(drop=True)

print(f"Distinct client position names: {len(position_inventory)}")
print(f"Groups with possible naming variations: {len(position_variations)}")
print("Comparison keys are for review; source names remain in the working dataset.")
display(position_inventory)
display(position_variations)

In [ ]:
# Step 13: Standardize the two reviewed spelling variations
labor_rates_df["position_original"] = labor_rates_df["position"].copy()
position_name_mapping = {
    # Changed: Plasterer Journeyperson -> Plasterer Journey Person.
    "Plasterer Journeyperson": "Plasterer Journey Person",
    # Changed: Sheet Metal Journeyperson -> Sheet Metal Journey Person.
    "Sheet Metal Journeyperson": "Sheet Metal Journey Person",
}
# All other client position names and their qualifiers are kept.
labor_rates_df["position"] = (
    labor_rates_df["position"].replace(position_name_mapping).astype("string")
)
changed_position_mask = labor_rates_df["position"].ne(
    labor_rates_df["position_original"]
).fillna(False)

# PDF matching must also map Journeyman to Journey Person; removing spaces is insufficient.
print(f"Position values standardized: {int(changed_position_mask.sum())}")
print(f"Distinct standardized positions: {labor_rates_df['position'].nunique()}")
display(labor_rates_df.loc[
    changed_position_mask,
    ["source_csv_row_number", "subcontractor", "position_original", "position"],
])

In [ ]:
# Step 14: Check for duplicates across every business field in the cleaned tracker
duplicate_comparison_columns = [
    column for column in labor_rates_df.columns
    if column != "source_csv_row_number"
]
all_duplicate_members = labor_rates_df.duplicated(
    subset=duplicate_comparison_columns, keep=False
)
redundant_record_count = int(labor_rates_df.duplicated(
    subset=duplicate_comparison_columns, keep="first"
).sum())
duplicate_records_review = labor_rates_df.loc[all_duplicate_members].copy()

print(f"Business columns compared: {len(duplicate_comparison_columns)}")
print(f"Rows belonging to duplicate groups: {len(duplicate_records_review)}")
print(f"Redundant copies after the first record: {redundant_record_count}")
display(duplicate_records_review)

In [ ]:
# Step 15: Create the condensed reference table for labor-rate validation.
cleaned_raw_labor_rates_tracker_df = labor_rates_df.copy(deep=True)

# Select the fields needed to match submitted PDF rates to schedules
validation_columns = [
    "source_csv_row_number",             # Evidence link to the original Labor_rate_data.csv record.
    "subcontractor",                    # Contractor whose approved schedule must be retrieved.
    "position",                         # Labor classification, including named-worker qualifiers.
    "effective_time_period_start",      # Start of the schedule's applicability to work dates.
    "effective_time_period_finish",     # End of the schedule's applicability to work dates.
    "reg_rate",                #regular-time hourly rate.
    "ot_rate",                 # Approved overtime hourly rate.
    "dt_rate",                 # Approved double-time hourly rate.
    "status",                           # Distinguishes Approved from pending or withdrawn schedules.
    "bu_approval_received",             # Approval history for reviewer evidence.
]
labor_rates_fa_df = cleaned_raw_labor_rates_tracker_df[validation_columns].copy()

# Missing values remain null for the future validator
# Source date issues remain visible; the validator must require a valid period before using a rate.
# Contacts, notes, premiums, and submission fields remain available in the full cleaned tracker.
print(f"Full cleaned tracker: {cleaned_raw_labor_rates_tracker_df.shape[0]} rows x {cleaned_raw_labor_rates_tracker_df.shape[1]} columns")
print(f"Validation reference table: {labor_rates_fa_df.shape[0]} rows x {labor_rates_fa_df.shape[1]} columns")
display(labor_rates_fa_df.head())

In [ ]:
# Step 16: export
export_tables = [
    (CLEAN_TRACKER_PATH, cleaned_raw_labor_rates_tracker_df),
    (VALIDATION_RATES_PATH, labor_rates_fa_df),
]
for output_path, table in export_tables:
    table.to_csv(
        output_path, index=False, date_format="%Y-%m-%d",
        float_format="%.2f", na_rep="",
    )
    date_fields = [column for column in date_columns if column in table.columns]
    reload_types = {
        column: str(dtype) for column, dtype in table.dtypes.items()
        if column not in date_fields
    }
    reloaded = pd.read_csv(
        output_path, dtype=reload_types, parse_dates=date_fields,
        keep_default_na=False, na_values=[""],
    )
    # Restore datetime resolution as well as missing dates for a precise schema check.
    for column in date_fields:
        reloaded[column] = pd.to_datetime(
            reloaded[column], format="%Y-%m-%d"
        ).astype(table[column].dtype)
    pd.testing.assert_frame_equal(
        table, reloaded, check_exact=False, rtol=0, atol=1e-10
    )
    print(f"Export verified: {output_path.relative_to(PROJECT_ROOT)} ({len(table)} rows, {len(table.columns)} columns)")